# Train Deeplabv3Plus with Custom Dataset
<a target="_blank" href="https://colab.research.google.com/github/SonySemiconductorSolutions/aitrios-rpi-tutorials-ai-model-training/blob/main/notebooks/deeplab3-pothole/custom_deeplab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

We will train a segmentation model on a [open-source pothole dataset](https://universe.roboflow.com/sankritya-rai-cldft/roadvis-segmentation/dataset/2). 

This tutorial is using training tools from https://github.com/SonySemiconductorSolutions/aitrios-rpi-training-samples. Specifically tools termed "deeplab3" which are for training of a [DeeplabV3Plus model](https://github.com/VainF/DeepLabV3Plus-Pytorch).

Tutorial includes:
- Installation of training repo
- Dataset setup
- Training setup
- Training and quantization
- Visualization
- Conversion


In [ ]:
# Test that we are running correct Colab runtime type
# If error, change Colab runtime to 2025.07
import sys
v = sys.version_info
assert v.major == 3 and v.minor == 11, "Python version must be 3.11"

# Installation
We start by cloning the training repo and install the package imx500_zoo. However, since we are working in Colab, we do not install the dependencies in pyproject.yaml (--no-deps). Instead, in the next cell, we will install a somewhat modified set of dependencies that have been tested in Colab.

In addition to the packages needed for training, we also install:
- Application Module Library (modlib) for visualization
- the converter tool, imx500-converter
- dataset loader from Roboflow

__OBSERVE 1__: When running through Colab, you have to restart the session after the installations in the next code block.

__OBSERVE 2__: In Colab you will get a message such as `ERROR: pip's dependency resolver ...`. You can ignore the error and continue to run the notebook (after having restarted the Colab session).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # clone repo
    !git clone --depth 1 https://github.com/SonySemiconductorSolutions/aitrios-rpi-training-samples; \
    
    # install imx500_zoo (without deps)
    !cd aitrios-rpi-training-samples && pip install --no-deps -e .
    !pip show imx500_zoo
    
    # install nanodet repo (required in order to use the training repo)
    !cd aitrios-rpi-training-samples/third_party/nanodet/nanodet && pip install -e .
    
    # Install training deps that works with Colab.
    # Install aitrios-rpi-application-module-library (modlib) for inference and visualization
    # Install imx500 converter so that we can convert the keras model to imx500 format.
    # Install roboflow for dataset handling
    !pip install \
    model-compression-toolkit==2.1.0 \
    torch==2.1.2 \
    torchvision==0.16.2 \
    torchinfo==1.8.0 \
    lightning==2.1.3 \
    onnx==1.14.1 \
    onnxruntime==1.15.1 \
    onnxruntime-extensions==0.10.1 \
    tensorflow~=2.15.0 \
    tensorflow-model-optimization==0.7.5 \
    tensorflow-addons==0.23.0 \
    pycocotools==2.0.7 \
    albumentations==1.3.1 \
    tabulate==0.9.0 \
    tqdm~=4.67.0 \
    "numpy<2.0" \
    safetensors~=0.5.0 \
    imx500-converter[tf] \
    git+https://github.com/SonySemiconductorSolutions/aitrios-rpi-application-module-library.git \
    roboflow
else:
    print("Not Colab - assuming container has necessary dependencies installed")

In [ ]:
"""
Perform initial checks in order to continue

if you get:
    RecursionError: maximum recursion depth exceeded while getting the repr of an object

then restart the session and re-run the cell
"""
import os
import shutil
import tensorflow as tf
import torch
import numpy as np
import modlib
import nanodet
import model_compression_toolkit
import roboflow

assert '2.15' in tf.__version__, print(tf.__version__)
assert '2.1.' in torch.__version__, print(torch.__version__)
assert '1' in np.__version__[0], print(np.__version__)

gpus = tf.config.experimental.list_physical_devices('GPU')
print("Num GPUs Available for TensorFlow: ", len(gpus))
if len(gpus)<1:
    print(f"WARNING: Found no GPU, running with GPU is highly recommended")

In [ ]:
# Converter requires java
import os
import re
import subprocess

def install_java(package: str = 'openjdk-17-jdk', version: int = 17) -> bool:
    try:
        result = subprocess.run(['java', '--version'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        version_output = result.stdout.splitlines()[0]
        match = re.search(r'(\d+)\.(\d+)\.(\d+)', version_output)  # Match version in form major.minor.patch
        print(f"Found Java version: {match.group(0)}")
        if match:
            major_version = int(match.group(1))
            if major_version == version:
                return True
            else:
                print(f"Java {version} is not installed. Installing correct version...")
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f"Java not installed. Installing...")

    try:
        is_root = os.geteuid() == 0
        prefix = [] if is_root else ['sudo']
        with open(os.devnull, 'w') as devnull:
            subprocess.run(prefix + ['apt', 'update'], check=True, stdout=devnull, stderr=devnull)
            subprocess.run(prefix + ['apt', 'install', '-y', package], check=True, stdout=devnull, stderr=devnull)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Installation error: {e}")
        return False

if install_java():
    print(f'Java installed')
else:
    print(f'Java missing and installation failed')

In [ ]:
# Settings
DATASET_NAME = 'RoadVis-Segmentation--2'
DATASET_PATH = f'aitrios-rpi-training-samples/samples/data/pothole'
QUANTIZED_MODEL = "aitrios-rpi-training-samples/samples/model/deeplab_v3p_pothole/deeplab_v3p_pothole_quantized.keras"
SHARED = "tutorial"

# Dataset
- Run the below cell and follow the link Roboflow where it will ask you to login to get an API key. 
- Once you have the API key you can enter it into the entry box that appeared when you ran the cell. 
- This will then download and unzip the dataset for you.

## Folder Structure
To use a custom dataset with DeepLab, you need to reconfigure the structure of the Dataset to fit the standard folder structure for DeepLabv3 training.
```
Datasets
└── train
      ├── JPEGimages
      ├── SegmentationClassRaw
      └── label_data.txt
    valid
      ├── JPEGimages
      ├── SegmentationClassRaw
      └── label_data.txt
    test
      ├── JPEGimages
      ├── SegmentationClassRaw
      └── label_data.txt
```
* The JPEGimages folder consists images in JPEG format with ".jpg" extention.
* The SegmentationClassRaw folder consists of labeled mask images in PNG format with ".png" extention.
* The label_data.txt file lists file names without extensions, one per line. Based on a single file name, it references two files: one JPEG file and one PNG file. For example, if label_data.txt contains "001", it references the files JPEGimages/001.jpg and SegmentationClassRaw/001.png.

In [ ]:
import os
from pathlib import Path
import roboflow

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    WORKDIR = Path("/content/")
else:
    WORKDIR = Path("/home/newuser/")

os.environ["WORKDIR"] = str(WORKDIR)

# Check if dataset already exists, if not download it and move to correct location
if Path(f'{DATASET_NAME}').exists():
    print(f"Dataset already downloaded: {DATASET_NAME}")
else:
    if Path(f'{DATASET_PATH}/{DATASET_NAME}').exists():
        print(f"Dataset already in correct location: {DATASET_PATH}/{DATASET_NAME}")
    else:
        # Login to Roboflow
        roboflow.login()

        # Download dataset
        rf = roboflow.Roboflow()
        project = rf.workspace("sankritya-rai-cldft").project("roadvis-segmentation")
        version = project.version(2)
        dataset = version.download("png-mask-semantic")    

In [ ]:
import os
import shutil

def create_directory(path):
    if not os.path.exists(path):
        os.makedirs(path)

def rename_file(file_path, new_name):
    directory = os.path.dirname(file_path)
    new_file_path = os.path.join(directory, new_name)
    os.rename(file_path, new_file_path)
    return new_file_path

def move_files(src_dir, dest_dir, extensions, rename_png=False):
    log_entries = []
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.lower().endswith(extensions):
                src_file_path = os.path.join(root, file)

                # Rename .png files if specified
                if rename_png and file.lower().endswith('.png'):
                    new_name = str(file[:-9]) + '.png' # Remove last 4 characters from the name
                    src_file_path = rename_file(src_file_path, new_name)

                dest_file_path = os.path.join(dest_dir, os.path.basename(src_file_path))
                shutil.move(src_file_path, dest_file_path)
                log_entries.append(str(file)[:-4])
    return log_entries

data_root = './RoadVis-Segmentation--2'
dest_root = './aitrios-rpi-training-samples/samples/data/pothole'
sets = ['train', 'valid', 'test']
for set_name in sets:
    src_dir = os.path.join(data_root, set_name)
    png_dir = os.path.join(dest_root, f'{set_name}/SegmentationClassRaw')
    jpg_dir = os.path.join(dest_root, f'{set_name}/JPEGimages')
    log_file_path = os.path.join(dest_root, f'{set_name}/label_data.txt')

    create_directory(png_dir)
    create_directory(jpg_dir)

    png_log_entries = move_files(src_dir, png_dir, ('.png',), rename_png=True)
    jpg_log_entries = move_files(src_dir, jpg_dir, ('.jpg', '.jpeg'))

    with open(log_file_path, 'w') as log_file:
        for entry in jpg_log_entries:
            log_file.write(entry + '\n')
    print(f'{set_name} has now been formatted')

In [ ]:
# Move README.dataset.txt to the dataset folder
from pathlib import Path
DATASET_PATH = 'aitrios-rpi-training-samples/samples/data/pothole'
if not Path(f'{DATASET_PATH}/README.dataset.txt').exists():
    !mv ./RoadVis-Segmentation--2/*txt $DATASET_PATH/

# Training configs
We will create two config files to train the model with our dataset. Main reference for this is the [training docs](https://github.com/SonySemiconductorSolutions/aitrios-rpi-training-samples/blob/main/docs/Deeplab_NewDataset.md).

## deeplab_v3p_pothole.ini
This is the primary config file. Note that we have updated:
* NUM_CLASSES, our model has 2 classes. NUM_CLASSES is always the total classes + 1
* BATCH_SIZE, set to 6 but can be adjusted, depends on your training setup
* NUM_EPOCHS, high value, observe that the trainer has early-stopping, i.e. will stop when model training does not improve after a certain amount of epochs. For this dataset, ~75 epochs are needed to get some proper segementation detections
* CONFIG, we have a custom model config file, see next

## deeplab_v3p_pothole.yml
This config file is used to configure the model. The main parameters:
* MODEL_PARAM, Cahneg the classes to the new class IDs and names
** ID: 0, is always background
** ID: N + 1, the last class is always viod
*DATA_ROOTPATH, update the path to the custom dataset location   



In [ ]:
%%bash
touch aitrios-rpi-training-samples/samples/deeplab_v3p_pothole.ini
cat <<EOF > aitrios-rpi-training-samples/samples/deeplab_v3p_pothole.ini

[SOLUTION]
FRAMEWORK = keras
RETRAIN = True
VALIDATE = True

[MODEL]
NAME = DeeplabV3p
INPUT_SIZE = [120, 120]
NUM_CLASSES = 2

[DATASET]
NAME = DefaultDeeplab

[TRAINER]
NAME = DeeplabV3pTrainer
BATCH_SIZE = 6
NUM_EPOCHS = 200 
LEARNING_RATE = 2e-2
CONFIG = ./config/deeplab_v3p_pothole.yml

[QUANTIZER]
NAME = MctKerasDeeplab

[VALIDATOR]
NAME = KerasSegmentationValidator
EOF

In [ ]:
%%bash
touch aitrios-rpi-training-samples/samples/config/deeplab_v3p_pothole.yml
cat <<EOF > aitrios-rpi-training-samples/samples/config/deeplab_v3p_pothole.yml
---
    backbone: 'mobilenetv2' 

    train_param: {
        gpu_id: 0
    }

    model_name: 'original'
    model_param: {
        classes: {
            0: "background", # first is always background
            1: "pothole",
            2: "void", # last is always void, as len() = num_class + 1
        }
    }

    data_rootpath: "${WORKDIR}/aitrios-rpi-training-samples/samples/data/pothole"

EOF

In [ ]:
%%bash
cat <<EOF > aitrios-rpi-training-samples/src/imx500_zoo/datasets/__init__.py

from imx500_zoo.datasets.cifar10 import Cifar10
from imx500_zoo.datasets.card_classification import CardClassification
from imx500_zoo.datasets.card_detection import CardDetection
from imx500_zoo.datasets.albu_imagefolder import AlbuImageFolder
from imx500_zoo.datasets.coco_val2017 import CocoVal2017
from imx500_zoo.datasets.imagenet import ImageNet
from imx500_zoo.datasets.arrow_posenet import ArrowPosenet
from imx500_zoo.datasets.default_posenet import DefaultPosenet
from imx500_zoo.datasets.default_deeplab import DefaultDeeplab
from imx500_zoo.datasets.card_segmentation import CardSegmentation
EOF

In [ ]:
%%bash
touch aitrios-rpi-training-samples/src/imx500_zoo/datasets/default_deeplab.py
cat <<EOF > aitrios-rpi-training-samples/src/imx500_zoo/datasets/default_deeplab.py

from imx500_zoo.utilities.deeplab_v3p.segmentation import Segmentation
from imx500_zoo.quantizers.mct_keras import get_representative_dataset
from imx500_zoo.quantizers.mct_keras_deeplab import CustomDataset
from imx500_zoo import utilities
import os


class DefaultDeeplab:
    DOWNLOAD_DATASET = None
    ZIP_SUBFOLDER = None
    ZIP_FILENAME = None
    def __init__(self, config):
        self.ini = config

        self.trainloader = None
        self.validloader = None
        self.dataloader_quant = None
        self.dataloader_eval = None

    def setup(self):
        self.config = self.ini.deeplab.config
        self.nclass = self.config.n_classes
        self.batchsize = self.config.batch_size
        self.imagesize = self.config.image_size
        self.d_root = self.config.data_rootpath

        self._download_dataset()
        self._setup_train()
        self._setup_quant()
        self._setup_valid()

    def get_loaders(self):
        return (
            self.trainloader,
            self.validloader,
            self.dataloader_quant,
            self.dataloader_eval,
        )

    def _download_dataset(self):
        if self.DOWNLOAD_DATASET is None:
            print(f"No download dataset provided for {self.__class__.__name__}, Skipping download.")
            return
        if self.ZIP_FILENAME is None:
            raise ValueError(f"ZIP_FILENAME is not set for {self.__class__.__name__}")
        if self.ZIP_SUBFOLDER is None:
            raise ValueError(f"ZIP_SUBFOLDER is not set for {self.__class__.__name__}")

        utilities.download_zip(
            self.DOWNLOAD_DATASET,
            self.data_path,
            self.ZIP_SUBFOLDER,
            target_name=self.ZIP_FILENAME,
        )

    def _setup_train(self):
        # Define the train and valid data generator.
        self.trainloader = Segmentation(
            config=self.config,
            dir=self.d_root,
            batch_size=self.batchsize,
            resize_shape=self.imagesize,
            blur=5,
            crop_shape=None,
            mode="train",
            n_classes=self.nclass,
            h_flip_en=True,
            v_flip_en=False,
            brightness=0.3,
            rotation=180,
            zoom=0.1,
            seed=7,
            contrast_en=False,
        )

        self.validloader = Segmentation(
            config=self.config,
            dir=self.d_root,
            batch_size=self.batchsize,
            resize_shape=self.imagesize,
            blur=0,
            crop_shape=None,
            mode="valid",
            n_classes=self.nclass,
            h_flip_en=True,
            v_flip_en=False,
            brightness=0.1,
            rotation=False,
            zoom=0.05,
            seed=7,
            contrast_en=False,
        )

    def _setup_quant(self):
        print(f"load dataset of quant : {self.config.rep_dataset}")
        dataset = CustomDataset(
            base_path=self.config.rep_dataset,
            img_size=self.config.image_width,
            batch_size=6,
        )
        self.dataloader_quant = get_representative_dataset(dataset, is_shuffle=True)

    def _setup_valid(self):
        self.dataloader_eval = Segmentation(
            config=self.config,
            dir=self.d_root,
            batch_size=1,
            resize_shape=self.imagesize,
            crop_shape=None,
            mode="test",
            n_classes=self.nclass,
            h_flip_en=False,
            v_flip_en=False,
            brightness=0,
            rotation=False,
            zoom=0,
            seed=7,
            contrast_en=False,
        )
EOF

# Training and Quantization
So, now we are ready to train the model.

* num_epochs is set to a fairly low value initially that does not give the best results but still detects the gauge meter. For good detection results, set this value to 1000. Training will end using early-stopping when learning stops to improve.

* ⚠️ If error occurs when importing mct after training, restart the session to fix the issue and make sure all global variables are redeclared. ⚠️

In [ ]:
# train and quantize
import os
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
!cd aitrios-rpi-training-samples/samples && imx500_zoo deeplab_v3p_pothole.ini

In [ ]:
assert os.path.exists(QUANTIZED_MODEL), f"Quantized file not found"

In [ ]:
""" Test that model_compression_toolkit (mct) still can be loaded after training

If mct fails to load:
- restart kernel
- reload settings code cell in the beginning of the notebook
"""
import importlib

module = "model_compression_toolkit"
try:
    importlib.import_module(module)
except Exception as e:
    print(f"failed import: {module} -> {e}")
    raise AssertionError(f"failed import: {module}")

# Visualize detections with [Aitrios Sample Apps](https://github.com/SonySemiconductorSolutions/aitrios-rpi-sample-apps)
Aitrios Sample Apps is a collection of applications running on the [Raspberry Pi AI camera](https://www.raspberrypi.com/documentation/accessories/ai-camera.html). It is using [Application Module Library](https://github.com/SonySemiconductorSolutions/aitrios-rpi-application-module-library) as its primary python library for application development. The gauge-monitoring app can also be used to visualize detections using Keras interpreter for inference, i.e. running inference without the sensor.



In [ ]:
"""visualize.py"""
from datetime import datetime
import os
from pathlib import Path
from typing import List
import shutil

import cv2
import numpy as np

from modlib.devices import AiCamera, KerasInterpreter
from modlib.devices.sources import Images
from modlib.apps import Annotator
from modlib.models import COLOR_FORMAT, Model, MODEL_TYPE
from modlib.models.results import Poses
from modlib.models.results import Segments

# When running on RPi with AI Camera
MODEL_PACKED =  Path.home() / Path("models/deeplab/pothole/pack/network.rpk")

class DeepLabv3(Model):
    def __init__(self, weights, model_type=MODEL_TYPE.RPK_PACKAGED, is_quantized: bool = True):
        print(f"Using model: {weights}")
        super().__init__(
            model_file=weights,
            model_type=model_type,
            color_format=COLOR_FORMAT.RGB,
            preserve_aspect_ratio=False,
        )

        self.in_width = 120
        self.in_height = 120
        self.is_quantized = is_quantized

    def pre_process(self, img):
        img = cv2.resize(img, (self.in_width, self.in_height), interpolation=cv2.INTER_AREA)
        in_tensor = img / 256.0
        in_tensor = np.expand_dims(in_tensor, axis=0)
        return (img, in_tensor)

    def post_process(self, output_tensors: List[np.ndarray]) -> Poses:
      new_mask = np.argmax(output_tensors[0],axis=2)
      return Segments(mask=np.where(new_mask == 0, -1, new_mask))

class MyKerasInterpreter(KerasInterpreter):
    def __init__(self, source, headless=False, timeout=None):
        super().__init__(source=source, headless=headless, timeout=timeout)

    @staticmethod
    def load_tf_keras_model(model: Model):
        """
        Loads a keras model file using tensorflow or model_compression_toolkit (if is_quantized is True).
        Requires tensorflow and model_compression_toolkit to be installed.

        Raises:
            ImportError: When loading the model fails due to missing dependencies.
        """
        model_path = model.model_file
        is_quantized = model.is_quantized

        try:
            print(f"Loading model: {model_path}, is_quantized: {is_quantized}")
            if is_quantized:
                import model_compression_toolkit as mct
                return mct.keras_load_quantized_model(model_path)
            else:
                import tensorflow as tf
                return tf.keras.models.load_model(model_path)
        except ImportError as e:
            raise ImportError(f"Failed to import model_compression_toolkit or tensorflow: {e}")
        except Exception as e:
            raise RuntimeError(f"Failed to load keras model: {e}")

    def deploy(self, model: Model):
        """
        Custom deploy method for KerasInterpreter.
        """
        print(f"[MyKerasInterpreter] Deploying model: {model.model_file}, {model.model_type}")
        if model.model_file is None:
            raise FileNotFoundError("Model file not found. Please provide a valid model file.")
        if not model.model_file.lower().endswith((".keras")):
            raise TypeError("Model file must be a Keras model file (.keras)")
        if not model.model_type == MODEL_TYPE.KERAS:
            raise TypeError("Model type must be Keras.")

        # Require model to have pre_process method
        if not hasattr(model, "pre_process"):
            raise AttributeError("Model must have a pre_process method to use MyKerasInterpreter.")

        self.model = model  # NOTE: deeplabv3 model has been modified to have is_quantized attribute
        self.keras_model = self.load_tf_keras_model(self.model)

def frame2image(img, output_dir="saved_images"):
    """Save the frame image with timestamp and gauge value."""
    # Generate timestamp for unique filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]  # Include milliseconds

    # Create filename
    filename = f"pothole_{timestamp}.jpg"
    filepath = os.path.join(output_dir, filename)
    print(filepath)
    # Save the image
    cv2.imwrite(filepath, img)
    print(f"Saved image: {filepath}")

    return filepath

def visualize(
    model_file,
    is_quantized=True,
    images=None,
    output_dir=None,
    threshold=0.4,
    save_image=False,
    debug=False):

    # Check files
    if not Path(model_file).exists():
        raise FileNotFoundError(f"Model file not found: {model_file}")

    # When we use keras model, we use MyKerasInterpreter and pre-recorded images as source
    if ".keras" in str(model_file):
        src = Images(images)
        device = MyKerasInterpreter(source=src)
        model = DeepLabv3(
            weights=model_file,
            model_type=MODEL_TYPE.KERAS,
            is_quantized=is_quantized)
    else:
        device = AiCamera()
        model = DeepLabv3(
            weights=model_file)

    device.deploy(model)

    annotator = Annotator()

    # Create output directory if it doesn't exist. Re-create existing directory
    if output_dir:
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        os.makedirs(output_dir)

    with device as stream:
        for frame in stream:
            if frame.detections.n_segments == 0:
                if save_image:
                    frame2image(frame.image)  # Save with value 0 for no detections
                else:
                    frame.display()
                if debug:
                    print("No detections")
                continue

            # apply filter
            annotator.annotate_segments(frame=frame, segments = frame.detections)

            if save_image:
                frame2image(frame.image)
            else:
                frame.display()

# This is executed automatically on RPi
if Path.home() == Path('/home/pi'):
    visualize(
        MODEL_PACKED)

In [ ]:
validation_images_folder = WORKDIR / Path("aitrios-rpi-training-samples/samples/data/pothole/valid/JPEGimages")
output_dir = WORKDIR / Path("saved_images")
visualize(QUANTIZED_MODEL, images=validation_images_folder,is_quantized=True, output_dir=output_dir, save_image=True)

In [ ]:
import os
from PIL import Image
from IPython.display import display

folder_path = output_dir

image_files = sorted([
    f for f in os.listdir(folder_path)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
])
assert len(image_files) > 0, "No annotated files from keras interpreter"
count = 0
for file_name in image_files:
  if count >= 10:
      break
  img_path = os.path.join(folder_path, file_name)
  img = Image.open(img_path)
  display(img)
  count += 1

# Conversion
For details see
* [Raspberry Pi Documentation](https://www.raspberrypi.com/documentation/accessories/ai-camera.html#conversion)
* [Sony IMX500 Converter documentation](https://developer.aitrios.sony-semicon.com/en/raspberrypi-ai-camera/documentation/imx500-converter)

In [ ]:
!imxconv-tf -i {QUANTIZED_MODEL} -o converted
! scp converted/packerOut.zip tutorial/
! scp aitrios-rpi-training-samples/samples/model/deeplab_v3p_pothole/deeplab_v3p_pothole_quantized.keras tutorial/

In [ ]:
"""
Expected output from converter:

dnnParams.xml		 deeplab_v3p_pothole_quantized_MemoryReport.json
deeplab_v3p_pothole_quantized.pbtxt  packerOut.zip
"""
!ls converted
assert os.path.exists("converted/packerOut.zip"), f"Converted file not found"

# Running the model on Raspberry Pi AI Camera
__OBSERVE__: First, save the quantized model and the output from the conversion to your local machine. For packaging you will need the `packerOut.zip` file.

Next step is to package the model for IMX500, see [Raspberry Pi Documentation](https://www.raspberrypi.com/documentation/accessories/ai-camera.html#packaging)

## Inference on Raspberry Pi
* Copy the visualization code from the cell above to `/home/pi/work/visualize.py`
* Install modlib on the Raspberry Pi, see [modlib](https://github.com/SonySemiconductorSolutions/aitrios-rpi-application-module-library)
* `visualize.py` expects that the packaged model and the configuration file is saved in certain folders, see variables:
```
  MODEL_PACKED
  CONFIG_PATH
```
* Copy some images from the validation dataset over to the Pi. Open any of the images and point the camera at the image.
* Run the application from the terminal:
```
  python3 visualize.py
```

The image below is a screenshot from the Pi. The camera has been oriented to watch that part of the monitor we are displaying one of the validation images. On the right we see the application window (started by visualize.py). It shows the camera view with annotated segment mask detection as a yellow area that fills the pothole. On the left we see the terminal window from where the application visualize.py was started.

<p align="center">
  <img src="./assets/deeplab_rpi_screenshot.png" width="1200"/>
</p>